# 1. Imports and Data Loading

In [1]:
# Install packages.
!pip install pandas numpy sentence-transformers torch scikit-learn gensim

In [2]:
# Import libraries.
import json
import copy
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from gensim.corpora import Dictionary
from gensim.models import CoherenceModel
from IPython.display import display
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import davies_bouldin_score, silhouette_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import normalize
from torch.utils.data import DataLoader, TensorDataset

In [3]:
# Set random seeds and select GPU.
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_STATE)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

Device: cuda


In [4]:
# Load the segment-level dataset and prepare normalized text inputs for both methods.
from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir('/content/drive/MyDrive/airbnb_review_text_analytics')

segment_df = pd.read_csv("data/processed/segment_df.csv")

# Use normalized segment text for both modeling and evaluation.
texts_norm = segment_df["segment_norm"].tolist()
tokenized_texts = [text.split() for text in texts_norm]

display(segment_df.head())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


,segment_id,review_id,listing_id,segment_text,segment_norm
0,0,463567767,10412695,This place was great!,place great
1,1,463567767,10412695,The pub downstairs has awesome food,pub downstairs awesome food
2,2,463567767,10412695,and seriously the best ceasar I've ever had.,seriously good ceasar ve
3,3,463567767,10412695,The roof top patio was amazing,roof patio amazing
4,4,463567767,10412695,and a great spot to chill out and people watch.,great spot chill people watch


# 2. Shared Evaluation Helpers

In [5]:
# Calculate the share of unique top words across topics.
def compute_topic_diversity(topic_terms, top_n=10):
    top_words = []

    for words in topic_terms.values():
        if not words:
            continue
        top_words.extend(words[:top_n])

    if not top_words:
        return np.nan

    return len(set(top_words)) / len(top_words)


# Calculate c_v coherence from topic keywords and tokenized documents.
def compute_topic_coherence_cv(topic_terms, tokenized_docs, no_below=5, no_above=0.5):
    if not tokenized_docs:
        return np.nan

    dictionary = Dictionary(tokenized_docs)
    dictionary.filter_extremes(no_below=no_below, no_above=no_above)

    filtered_topics = []
    for words in topic_terms.values():
        keep_words = [word for word in words[:10] if word in dictionary.token2id]
        if keep_words:
            filtered_topics.append(keep_words)

    if len(filtered_topics) < 2:
        return np.nan

    coherence_model = CoherenceModel(
        topics=filtered_topics,
        texts=tokenized_docs,
        dictionary=dictionary,
        coherence="c_v",
    )
    return coherence_model.get_coherence()


# Calculate silhouette and Davies-Bouldin scores after removing outlier labels.
def compute_clustering_metrics(features, labels, sample_size=10000):
    labels = np.asarray(labels)
    valid_mask = labels != -1

    X_valid = np.asarray(features)[valid_mask]
    labels_valid = labels[valid_mask]

    if len(labels_valid) < 2 or len(np.unique(labels_valid)) < 2:
        return np.nan, np.nan

    sample_n = min(sample_size, len(labels_valid))
    if sample_n < len(labels_valid):
        rng = np.random.default_rng(RANDOM_STATE)
        sample_idx = rng.choice(len(labels_valid), size=sample_n, replace=False)
        X_eval = X_valid[sample_idx]
        labels_eval = labels_valid[sample_idx]
    else:
        X_eval = X_valid
        labels_eval = labels_valid

    silhouette = silhouette_score(X_eval, labels_eval)
    davies_bouldin = davies_bouldin_score(X_eval, labels_eval)
    return silhouette, davies_bouldin


# Combine all evaluation metrics into one method-level summary record.
def summarize_method(method_name, labels, topic_terms, tokenized_docs, features):
    labels = np.asarray(labels)
    non_outlier_labels = labels[labels != -1]

    outlier_rate = float(np.mean(labels == -1))
    number_of_topics = int(len(np.unique(non_outlier_labels))) if len(non_outlier_labels) else 0
    topic_diversity = compute_topic_diversity(topic_terms, top_n=10)
    topic_coherence = compute_topic_coherence_cv(topic_terms, tokenized_docs)
    silhouette, davies_bouldin = compute_clustering_metrics(features, labels)

    return {
        "method": method_name,
        "outlier_rate": outlier_rate,
        "number_of_topics": number_of_topics,
        "topic_diversity_at_10": topic_diversity,
        "topic_coherence_c_v": topic_coherence,
        "silhouette_score": silhouette,
        "davies_bouldin_index": davies_bouldin,
    }


# Save topic keyword dictionaries for later inspection.
def save_topic_terms(topic_terms, output_path):
    serializable = {str(key): list(value) for key, value in topic_terms.items()}
    Path(output_path).write_text(json.dumps(serializable, indent=2), encoding="utf-8")

# 3. Neural Topic Pipeline


In [6]:
# Encode normalized review segments with the same sentence-transformer family used in the BERTopic workflow.
embedding_model_name = "all-MiniLM-L6-v2"
embedder = SentenceTransformer(embedding_model_name)

embeddings = embedder.encode(
    texts_norm,
    batch_size=128,
    show_progress_bar=True,
    normalize_embeddings=True,
)
embedding_matrix = np.asarray(embeddings, dtype=np.float32)

print("Embedding matrix shape:", embedding_matrix.shape)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/299 [00:00<?, ?it/s]

Embedding matrix shape: (38145, 384)


In [7]:
# Define a compact autoencoder to learn lower-dimensional representations of sentence embeddings.
class AutoEncoder(nn.Module):
    def __init__(self, input_dim, latent_dim):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, latent_dim),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Linear(256, input_dim),
        )

    def forward(self, x):
        latent = self.encoder(x)
        reconstruction = self.decoder(latent)
        return latent, reconstruction


# Train one autoencoder configuration with validation loss and early stopping.
def fit_autoencoder(
    X_array,
    latent_dim,
    learning_rate,
    batch_size=256,
    epochs=8,
    patience=2,
    validation_fraction=0.1,
):
    X_train, X_val = train_test_split(
        X_array,
        test_size=validation_fraction,
        random_state=RANDOM_STATE,
    )

    train_tensor = torch.tensor(X_train, dtype=torch.float32)
    val_tensor = torch.tensor(X_val, dtype=torch.float32)

    train_loader = DataLoader(TensorDataset(train_tensor), batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(TensorDataset(val_tensor), batch_size=batch_size, shuffle=False)

    model = AutoEncoder(input_dim=X_array.shape[1], latent_dim=latent_dim).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    loss_fn = nn.MSELoss()

    best_state = None
    best_val_loss = np.inf
    epochs_without_improvement = 0
    history = []

    for epoch in range(1, epochs + 1):
        model.train()
        train_loss = 0.0

        for (batch,) in train_loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            _, reconstruction = model(batch)
            loss = loss_fn(reconstruction, batch)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * batch.size(0)

        train_loss /= len(train_tensor)

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for (batch,) in val_loader:
                batch = batch.to(device)
                _, reconstruction = model(batch)
                loss = loss_fn(reconstruction, batch)
                val_loss += loss.item() * batch.size(0)

        val_loss /= len(val_tensor)
        history.append(
            {
                "epoch": epoch,
                "train_loss": train_loss,
                "val_loss": val_loss,
            }
        )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= patience:
            break

    model.load_state_dict(best_state)
    return model, best_val_loss, pd.DataFrame(history)


# Encode all embeddings into the trained latent space and normalize the latent vectors.
def encode_latent(model, X_array, batch_size=512):
    tensor_X = torch.tensor(X_array, dtype=torch.float32)
    loader = DataLoader(TensorDataset(tensor_X), batch_size=batch_size, shuffle=False)

    latent_batches = []
    model.eval()
    with torch.no_grad():
        for (batch,) in loader:
            batch = batch.to(device)
            latent, _ = model(batch)
            latent_batches.append(latent.cpu().numpy())

    latent_matrix = np.vstack(latent_batches)
    return normalize(latent_matrix, norm="l2").astype(np.float32)

In [8]:
# Tune the autoencoder with a small parameter grid and a subset of segments.
ae_tuning_sample_size = min(4000, len(embedding_matrix))
rng = np.random.default_rng(RANDOM_STATE)
ae_tuning_idx = rng.choice(len(embedding_matrix), size=ae_tuning_sample_size, replace=False)
X_tune = embedding_matrix[ae_tuning_idx]

ae_search_space = [
    {"latent_dim": 16, "learning_rate": 1e-3},
    {"latent_dim": 32, "learning_rate": 1e-3},
    {"latent_dim": 64, "learning_rate": 1e-3},
    {"latent_dim": 32, "learning_rate": 5e-4},
]

ae_search_results = []
best_ae_model = None
best_ae_params = None
best_ae_history = None
best_ae_loss = np.inf

for params in ae_search_space:
    model, val_loss, history = fit_autoencoder(
        X_tune,
        latent_dim=params["latent_dim"],
        learning_rate=params["learning_rate"],
        epochs=8,
        patience=2,
    )

    result = {
        "latent_dim": params["latent_dim"],
        "learning_rate": params["learning_rate"],
        "best_val_loss": val_loss,
        "epochs_ran": int(history["epoch"].max()),
    }
    ae_search_results.append(result)

    if val_loss < best_ae_loss:
        best_ae_loss = val_loss
        best_ae_model = model
        best_ae_params = params
        best_ae_history = history

ae_tuning_results = pd.DataFrame(ae_search_results).sort_values("best_val_loss").reset_index(drop=True)
display(ae_tuning_results)
print("Best autoencoder params:", best_ae_params)

,latent_dim,learning_rate,best_val_loss,epochs_ran
0,32,0.0010,0.001430,8
1,16,0.0010,0.001450,8
2,64,0.0010,0.001458,8
3,32,0.0005,0.001622,8


Best autoencoder params: {'latent_dim': 32, 'learning_rate': 0.001}


In [9]:
# Refit the selected autoencoder configuration on the full embedding matrix and generate latent vectors.
final_ae_model, final_ae_val_loss, final_ae_history = fit_autoencoder(
    embedding_matrix,
    latent_dim=best_ae_params["latent_dim"],
    learning_rate=best_ae_params["learning_rate"],
    epochs=12,
    patience=3,
)

latent_matrix = encode_latent(final_ae_model, embedding_matrix)

display(final_ae_history.tail())
print("Final latent matrix shape:", latent_matrix.shape)

,epoch,train_loss,val_loss
7,8,0.000873,0.000876
8,9,0.000862,0.000869
9,10,0.000852,0.000860
10,11,0.000843,0.000854
11,12,0.000835,0.000846


Final latent matrix shape: (38145, 32)


In [10]:
# Select the K-Means cluster count with a lightweight silhouette-based search on the latent space.
# Candidate values cover a similar topic granularity to the BERTopic baseline.
cluster_candidates = [30, 35, 40, 45, 50, 55, 60, 65, 70, 75, 80]
cluster_tuning_sample_size = min(6000, len(latent_matrix))
cluster_idx = rng.choice(len(latent_matrix), size=cluster_tuning_sample_size, replace=False)
Z_cluster_sample = latent_matrix[cluster_idx]

cluster_search_results = []
best_cluster_k = None
best_cluster_score = -np.inf

for k in cluster_candidates:
    if len(Z_cluster_sample) <= k:
        continue

    candidate_kmeans = KMeans(n_clusters=k, n_init=20, random_state=RANDOM_STATE)
    candidate_labels = candidate_kmeans.fit_predict(Z_cluster_sample)

    silhouette = silhouette_score(Z_cluster_sample, candidate_labels)
    davies_bouldin = davies_bouldin_score(Z_cluster_sample, candidate_labels)

    cluster_search_results.append(
        {
            "n_clusters": k,
            "silhouette_score": silhouette,
            "davies_bouldin_index": davies_bouldin,
        }
    )

    candidate_score = silhouette - 0.05 * davies_bouldin
    if candidate_score > best_cluster_score:
        best_cluster_score = candidate_score
        best_cluster_k = k

cluster_tuning_results = pd.DataFrame(cluster_search_results).sort_values(
    ["silhouette_score", "davies_bouldin_index"],
    ascending=[False, True],
).reset_index(drop=True)

display(cluster_tuning_results)
print("Selected number of neural clusters:", best_cluster_k)

,n_clusters,silhouette_score,davies_bouldin_index
0,70,0.130772,2.130731
1,50,0.127144,2.138370
2,55,0.126462,2.165566
3,80,0.126230,2.185056
4,60,0.125347,2.186419
5,65,0.123482,2.189451
6,75,0.122492,2.198449
7,45,0.119615,2.237536
8,40,0.117855,2.252638
9,35,0.116129,2.182713


Selected number of neural clusters: 70


In [11]:
# Assign each segment to a neural topic using the selected K-Means model.
neural_kmeans = KMeans(n_clusters=best_cluster_k, n_init=50, random_state=RANDOM_STATE)
neural_labels = neural_kmeans.fit_predict(latent_matrix)

segment_df["nn_topic_id"] = neural_labels
segment_df["nn_outlier"] = 0

In [12]:
# Extract representative TF-IDF keywords for each neural topic.
nn_vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=5,
    max_features=20000,
)

nn_tfidf = nn_vectorizer.fit_transform(texts_norm)
nn_terms = np.array(nn_vectorizer.get_feature_names_out())

# Build a topic summary table with topic size and top keywords.
neural_topic_rows = []
neural_topic_terms = {}

for cluster_id in sorted(np.unique(neural_labels)):
    cluster_idx = np.where(neural_labels == cluster_id)[0]
    cluster_mean = nn_tfidf[cluster_idx].mean(axis=0).A1
    top_terms = nn_terms[np.argsort(-cluster_mean)[:10]].tolist()
    neural_topic_terms[int(cluster_id)] = top_terms

    neural_topic_rows.append(
        {
            "topic_id": int(cluster_id),
            "size": int(len(cluster_idx)),
            "top_keywords": ", ".join(top_terms),
        }
    )

neural_topic_summary = pd.DataFrame(neural_topic_rows).sort_values("topic_id").reset_index(drop=True)
display(neural_topic_summary)

,topic_id,size,top_keywords
0,0,621,"comfortable, clean comfortable, clean, place, ..."
1,1,383,"home, feel, feel home, right home, feel right,..."
2,2,780,"downtown, city, location, great, central, stre..."
3,3,872,"expect, expectation, need, exceed expectation,..."
4,4,813,"toronto, stay, stay toronto, visit toronto, vi..."
...,...,...,...
65,65,376,"picture, photo, exactly, look, place, like, ex..."
66,66,313,"recommend place, recommend, place, highly, hig..."
67,67,612,"location, convenient, easy, locate, place, acc..."
68,68,539,"parking, street, car, park, parking spot, gara..."


# 4. Neural Topic Evaluation

In [13]:
# Summarize the neural topic model with the shared evaluation metrics.
neural_metrics = summarize_method(
    method_name="neural_autoencoder_kmeans",
    labels=neural_labels,
    topic_terms=neural_topic_terms,
    tokenized_docs=tokenized_texts,
    features=latent_matrix,
)

neural_metrics_df = pd.DataFrame(
    {
        "metric": list(neural_metrics.keys())[1:],
        "value": list(neural_metrics.values())[1:],
    }
)
display(neural_metrics_df)

,metric,value
0,outlier_rate,0.000000
1,number_of_topics,70.000000
2,topic_diversity_at_10,0.611429
3,topic_coherence_c_v,0.526489
4,silhouette_score,0.123743
5,davies_bouldin_index,2.151791


In [14]:
# Build a topic dictionary table to make the Neural Network + K-Means results interpretable.
TOPIC_PHRASES_K = 10
REPRESENTATIVE_SEGMENTS_K = 3

def get_top_phrases_for_topic(docs, top_k=10):
    """
    Return the most frequent 2-word and 3-word phrases within one topic.
    """
    if len(docs) == 0:
        return []

    vectorizer = CountVectorizer(
        ngram_range=(2, 3),
        stop_words="english",
        min_df=2
    )

    try:
        phrase_matrix = vectorizer.fit_transform(docs)
    except ValueError:
        return []

    phrase_counts = phrase_matrix.sum(axis=0).A1
    phrases = vectorizer.get_feature_names_out()

    phrase_freq = pd.DataFrame({
        "phrase": phrases,
        "count": phrase_counts
    }).sort_values("count", ascending=False)

    return phrase_freq["phrase"].head(top_k).tolist()


def get_representative_segments_for_topic(cluster_id, top_k=3):
    """
    Return segments closest to the K-Means centroid as representative examples.
    """
    cluster_idx = np.where(neural_labels == cluster_id)[0]

    if len(cluster_idx) == 0:
        return []

    cluster_vectors = latent_matrix[cluster_idx]
    centroid = neural_kmeans.cluster_centers_[cluster_id]

    distances = np.linalg.norm(cluster_vectors - centroid, axis=1)
    closest_local_idx = np.argsort(distances)[:top_k]
    closest_global_idx = cluster_idx[closest_local_idx]

    return (
        segment_df.iloc[closest_global_idx]["segment_norm"]
        .dropna()
        .astype(str)
        .tolist()
    )


topic_dict_rows = []

for tid in sorted(segment_df["nn_topic_id"].unique()):
    tid = int(tid)

    topic_docs = (
        segment_df.loc[
            segment_df["nn_topic_id"] == tid,
            "segment_norm"
        ]
        .dropna()
        .astype(str)
        .tolist()
    )

    top_phrases = get_top_phrases_for_topic(
        topic_docs,
        top_k=TOPIC_PHRASES_K
    )

    reps = get_representative_segments_for_topic(
        tid,
        top_k=REPRESENTATIVE_SEGMENTS_K
    )

    topic_dict_rows.append({
        "topic_id": tid,
        "topic_label": f"Topic_{tid}",
        "top_phrases": ", ".join(top_phrases),
        "n_segments_topic": len(topic_docs),
        "representative_segments": " || ".join(reps)
    })


neural_topic_dictionary_table = pd.DataFrame(topic_dict_rows).sort_values(
    "n_segments_topic",
    ascending=False
).reset_index(drop=True)

neural_topic_dictionary_table.to_csv("data/processed/neural_topic_dictionary_table.csv", index=False)
print("Saved: neural_topic_dictionary_table.csv")

neural_topic_dictionary_table.head()

Saved: neural_topic_dictionary_table.csv


,topic_id,topic_label,top_phrases,n_segments_topic,representative_segments
0,53,Topic_53,"enjoy stay, love stay, recommend stay, highly ...",1069,have stay || glad choose stay || make sure go ...
1,47,Topic_47,"apartment clean, apartment great, great apartm...",1022,nice modern fresh apartment || place great act...
2,3,Topic_3,"exceed expectation, answer question, win disap...",872,turn exactly need || team go way fix || say ye...
3,36,Topic_36,"great location, location great, good location,...",862,great location || great location || great loca...
4,12,Topic_12,"subway station, public transportation, union s...",839,easily route 12 bus 5 minute walk b b kennedy ...


# 5. LDA Topic Pipeline


In [15]:
# Build count-based features for LDA using normalized segment text.
lda_vectorizer = CountVectorizer(
    stop_words="english",
    min_df=5,
    max_df=0.95,
    ngram_range=(1, 1),
    max_features=15000,
)

X_counts = lda_vectorizer.fit_transform(texts_norm)
lda_feature_names = lda_vectorizer.get_feature_names_out()

print("Document-term matrix shape:", X_counts.shape)

Document-term matrix shape: (38145, 2135)


In [16]:
# Extract the highest-weighted words from each LDA topic.
def extract_lda_topic_terms(lda_model, vocabulary, top_n=10):
    topic_terms = {}

    for topic_idx, topic in enumerate(lda_model.components_):
        top_words = vocabulary[np.argsort(topic)[-top_n:][::-1]].tolist()
        topic_terms[int(topic_idx)] = top_words

    return topic_terms

In [17]:
# Tune LDA over a small set of topic counts to keep the comparison practical.
# Candidate values cover a broader topic range for comparison with the BERTopic baseline.
lda_topic_candidates = [30, 35, 40, 45, 50, 55, 60, 65, 70, 75, 80]
lda_tuning_sample_size = min(5000, X_counts.shape[0])
lda_tuning_idx = rng.choice(X_counts.shape[0], size=lda_tuning_sample_size, replace=False)
X_counts_tune = X_counts[lda_tuning_idx]
tokenized_texts_tune = [tokenized_texts[i] for i in lda_tuning_idx]

lda_search_rows = []
best_lda_model = None
best_lda_terms = None
best_lda_params = None
best_lda_score = -np.inf

for n_topics in lda_topic_candidates:
    lda_candidate = LatentDirichletAllocation(
        n_components=n_topics,
        max_iter=10,
        learning_method="batch",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    lda_candidate.fit(X_counts_tune)

    candidate_topic_terms = extract_lda_topic_terms(lda_candidate, lda_feature_names, top_n=10)
    coherence = compute_topic_coherence_cv(candidate_topic_terms, tokenized_texts_tune)
    diversity = compute_topic_diversity(candidate_topic_terms, top_n=10)
    perplexity = lda_candidate.perplexity(X_counts_tune)

    lda_search_rows.append(
        {
            "n_topics": n_topics,
            "topic_coherence_c_v": coherence,
            "topic_diversity_at_10": diversity,
            "perplexity": perplexity,
        }
    )

    candidate_score = coherence + 0.1 * diversity
    if candidate_score > best_lda_score:
        best_lda_score = candidate_score
        best_lda_model = lda_candidate
        best_lda_terms = candidate_topic_terms
        best_lda_params = {"n_topics": n_topics}

lda_tuning_results = pd.DataFrame(lda_search_rows).sort_values(
    ["topic_coherence_c_v", "topic_diversity_at_10"],
    ascending=[False, False],
).reset_index(drop=True)

display(lda_tuning_results)
print("Selected LDA params:", best_lda_params)

,n_topics,topic_coherence_c_v,topic_diversity_at_10,perplexity
0,50,0.401023,0.802000,1114.931306
1,55,0.400649,0.812727,1156.927253
2,65,0.397912,0.807692,1237.459308
3,40,0.386081,0.795000,1061.878709
4,75,0.385895,0.785333,1249.565896
5,70,0.384015,0.784286,1246.348576
6,30,0.379972,0.793333,940.622987
7,60,0.370333,0.778333,1184.722940
8,45,0.370217,0.771111,1060.856028
9,35,0.368541,0.785714,1008.541034


Selected LDA params: {'n_topics': 55}


In [18]:
# Fit the final LDA model and create a topic summary table.
final_lda = LatentDirichletAllocation(
    n_components=best_lda_params["n_topics"],
    max_iter=15,
    learning_method="batch",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
final_lda.fit(X_counts)

lda_doc_topic = final_lda.transform(X_counts)
lda_labels = lda_doc_topic.argmax(axis=1)
lda_topic_terms = extract_lda_topic_terms(final_lda, lda_feature_names, top_n=10)

segment_df["lda_topic_id"] = lda_labels

lda_topic_rows = []
for topic_id in sorted(lda_topic_terms):
    lda_topic_rows.append(
        {
            "topic_id": int(topic_id),
            "size": int((lda_labels == topic_id).sum()),
            "top_keywords": ", ".join(lda_topic_terms[topic_id]),
        }
    )

lda_topic_summary = pd.DataFrame(lda_topic_rows).sort_values("topic_id").reset_index(drop=True)
display(lda_topic_summary)

,topic_id,size,top_keywords
0,0,591,"communication, issue, leave, town, arrival, wa..."
1,1,822,"good, awesome, place, wifi, ve, michael, far, ..."
2,2,694,"definitely, recommend, ll, let, breakfast, pla..."
3,3,584,"quiet, neighborhood, ask, safe, property, coul..."
4,4,1005,"recommend, highly, place, accommodation, tv, g..."
5,5,332,"come, help, car, cook, beach, canada, mall, me..."
6,6,332,"look, ve, pleasant, forward, beautifully, revi..."
7,7,429,"shop, coffee, food, use, local, fridge, snack,..."
8,8,665,"night, little, small, bit, floor, door, noise,..."
9,9,601,"apartment, family, basement, evening, prompt, ..."


# 6. LDA Evaluation

In [19]:
# Summarize the LDA model with the shared evaluation metrics.
lda_metrics = summarize_method(
    method_name="lda",
    labels=lda_labels,
    topic_terms=lda_topic_terms,
    tokenized_docs=tokenized_texts,
    features=lda_doc_topic,
)

lda_metrics_df = pd.DataFrame(
    {
        "metric": list(lda_metrics.keys())[1:],
        "value": list(lda_metrics.values())[1:],
    }
)
display(lda_metrics_df)

,metric,value
0,outlier_rate,0.000000
1,number_of_topics,55.000000
2,topic_diversity_at_10,0.861818
3,topic_coherence_c_v,0.358550
4,silhouette_score,0.194558
5,davies_bouldin_index,1.348845


In [20]:
# Build a topic dictionary table to make the LDA results interpretable.
TOPIC_PHRASES_K = 10
REPRESENTATIVE_SEGMENTS_K = 3

def get_top_phrases_for_topic(docs, top_k=10):
    """
    Return the most frequent 2-word and 3-word phrases within one topic.
    """
    if len(docs) == 0:
        return []

    vectorizer = CountVectorizer(
        ngram_range=(2, 3),
        stop_words="english",
        min_df=2
    )

    try:
        phrase_matrix = vectorizer.fit_transform(docs)
    except ValueError:
        return []

    phrase_counts = phrase_matrix.sum(axis=0).A1
    phrases = vectorizer.get_feature_names_out()

    phrase_freq = pd.DataFrame({
        "phrase": phrases,
        "count": phrase_counts
    }).sort_values("count", ascending=False)

    return phrase_freq["phrase"].head(top_k).tolist()


topic_dict_rows = []

for tid in sorted(np.unique(lda_labels)):
    tid = int(tid)

    topic_mask = segment_df["lda_topic_id"] == tid

    topic_docs = (
        segment_df.loc[topic_mask, "segment_norm"]
        .dropna()
        .astype(str)
        .tolist()
    )

    top_phrases = get_top_phrases_for_topic(
        topic_docs,
        top_k=TOPIC_PHRASES_K
    )

    topic_indices = np.where(topic_mask.to_numpy())[0]

    # Use the highest document-topic probabilities as representative segments.
    top_rep_indices = topic_indices[
        np.argsort(-lda_doc_topic[topic_indices, tid])[:REPRESENTATIVE_SEGMENTS_K]
    ]

    reps = (
        segment_df.iloc[top_rep_indices]["segment_norm"]
        .dropna()
        .astype(str)
        .tolist()
    )

    topic_dict_rows.append({
        "topic_id": tid,
        "topic_label": f"Topic_{tid}",
        "top_phrases": ", ".join(top_phrases),
        "n_segments_topic": int(topic_mask.sum()),
        "representative_segments": " || ".join(reps)
    })


lda_topic_dictionary_table = pd.DataFrame(topic_dict_rows).sort_values(
    "n_segments_topic",
    ascending=False
).reset_index(drop=True)

lda_topic_dictionary_table.to_csv("data/processed/lda_topic_dictionary_table.csv", index=False)
print("Saved: lda_topic_dictionary_table.csv")

lda_topic_dictionary_table.head()

Saved: lda_topic_dictionary_table.csv


,topic_id,topic_label,top_phrases,n_segments_topic,representative_segments
0,23,Topic_23,"great stay, definitely stay, place stay, wonde...",2136,joy stay le s place || overall great stay || o...
1,49,Topic_49,"great location, location great, location perfe...",1854,great unit great location || location great co...
2,26,Topic_26,"place clean, clean comfortable, super clean, a...",1848,vasily s place clean tidy || place clean tidy ...
3,22,Topic_22,"great place, place great, place stay, great pl...",1433,great value great price || great great place |...
4,14,Topic_14,"easy check, easy access, check easy, check che...",1196,easy check check process || check check proces...


# 7. Method Comparison Summary

In [21]:
# Combine method-level metrics into one comparison table and save the result.
bertopic_metrics_raw = pd.read_csv("data/processed/bertopic_evaluation_summary.csv")

bertopic_metrics = bertopic_metrics_raw.set_index("metric")["value"].to_dict()
bertopic_metrics["method"] = "bertopic"

comparison_summary = pd.DataFrame(
    [bertopic_metrics, neural_metrics, lda_metrics]
)[
    [
        "method",
        "outlier_rate",
        "number_of_topics",
        "topic_diversity_at_10",
        "topic_coherence_c_v",
        "silhouette_score",
        "davies_bouldin_index",
    ]
]

comparison_summary = comparison_summary.round(
    {
        "outlier_rate": 4,
        "number_of_topics": 0,
        "topic_diversity_at_10": 4,
        "topic_coherence_c_v": 4,
        "silhouette_score": 4,
        "davies_bouldin_index": 4,
    }
)

display(comparison_summary)

,method,outlier_rate,number_of_topics,topic_diversity_at_10,topic_coherence_c_v,silhouette_score,davies_bouldin_index
0,bertopic,0.2327,53.0,0.2245,0.3957,0.0728,2.9999
1,neural_autoencoder_kmeans,0.0000,70.0,0.6114,0.5265,0.1237,2.1518
2,lda,0.0000,55.0,0.8618,0.3586,0.1946,1.3488


### Method Comparison Insights

The three methods show different strengths. LDA achieved the strongest clustering-separation results, with the highest silhouette score and the lowest Davies-Bouldin index. It also had the highest topic diversity, suggesting that its top terms overlap less across topics. The neural autoencoder + K-Means method achieved the highest topic coherence score, meaning its topic terms were the most internally consistent based on the c_v metric. BERTopic had weaker quantitative scores, especially in topic diversity and clustering separation, and it also produced a 23.27% outlier rate because HDBSCAN does not force every segment into a topic.

However, the outlier rate should not be interpreted as purely negative. Unlike LDA and K-Means, BERTopic can identify uncertain or weakly clustered segments as outliers, which makes its assigned topics more conservative and potentially more reliable for interpretation.

### Why BERTopic Was Selected

Although LDA and the neural autoencoder + K-Means method performed better on several quantitative metrics, BERTopic was selected as the main method because its topic dictionary produced more interpretable and business-relevant themes. The BERTopic topics were easier to map to meaningful Airbnb review categories such as host quality, check-in, and location convenience.

In comparison, some LDA and neural topics had stronger metric scores but were more general or repetitive, often centered around broad phrases such as “great location”, “great place”, “good stay”, or “highly recommend”. These topics are useful statistically but harder to translate into clear dashboard categories or actionable business insights.